# Proximal Policy Optimization (PPO) on CartPole

CSCI 6353 · Topic 39.

PPO limits how much the policy can change in a single update, using a **clipped surrogate**
objective. It keeps TRPO's trust-region idea but replaces the hard constraint with a one-line
clip. Paired with **GAE** advantages and **K-epoch data reuse**, it is the most widely used
deep-RL algorithm today. Runs on **CPU**. On Colab: *Runtime → Run all*.

## 1. Setup

In [ ]:
!pip -q install gymnasium torch matplotlib

In [ ]:
import gymnasium as gym
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.distributions import Categorical
import matplotlib.pyplot as plt

learning_rate = 0.0005
gamma     = 0.98
lmbda     = 0.95
eps_clip  = 0.1
K_epoch   = 3
T_horizon = 20

## 2. The PPO network

Same shared-body actor-critic as Topic 38, now with three new pieces in `train_net`: the
**probability ratio** $r_t = \pi_{new}/\pi_{old}$ (we stored the old prob), the **GAE**
advantage (backward recursion with $\gamma\lambda$), and the **clipped** loss
$-\min(r_t A_t,\ \mathrm{clip}(r_t,1-\epsilon,1+\epsilon)A_t)$. The `K_epoch` loop reuses
each batch several times — safe precisely because the clip keeps every step small.

In [ ]:
class PPO(nn.Module):
    def __init__(self):
        super().__init__()
        self.data = []
        self.fc1   = nn.Linear(4, 256)
        self.fc_pi = nn.Linear(256, 2)     # actor head
        self.fc_v  = nn.Linear(256, 1)     # critic head
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def pi(self, x, softmax_dim=0):
        x = F.relu(self.fc1(x))
        return F.softmax(self.fc_pi(x), dim=softmax_dim)

    def v(self, x):
        x = F.relu(self.fc1(x))
        return self.fc_v(x)

    def put_data(self, transition):
        self.data.append(transition)

    def make_batch(self):
        s, a, r, s2, prob_a, done = zip(*self.data)
        s  = torch.tensor(s,  dtype=torch.float)
        a  = torch.tensor(a).unsqueeze(1)
        r  = torch.tensor(r,  dtype=torch.float).unsqueeze(1)
        s2 = torch.tensor(s2, dtype=torch.float)
        prob_a = torch.tensor(prob_a, dtype=torch.float).unsqueeze(1)
        done_mask = torch.tensor([[0.0 if d else 1.0] for d in done])
        self.data = []
        return s, a, r, s2, done_mask, prob_a

    def train_net(self):
        s, a, r, s2, done_mask, prob_a = self.make_batch()
        for _ in range(K_epoch):                       # reuse the same batch
            td_target = r + gamma * self.v(s2) * done_mask
            delta = (td_target - self.v(s)).detach().numpy()
            adv_lst, adv = [], 0.0                      # GAE, computed backward
            for d in delta[::-1]:
                adv = gamma * lmbda * adv + d[0]
                adv_lst.append([adv])
            adv_lst.reverse()
            advantage = torch.tensor(adv_lst, dtype=torch.float)

            pi = self.pi(s, softmax_dim=1)
            pi_a = pi.gather(1, a)
            ratio = torch.exp(torch.log(pi_a) - torch.log(prob_a))   # pi_new / pi_old
            surr1 = ratio * advantage
            surr2 = torch.clamp(ratio, 1 - eps_clip, 1 + eps_clip) * advantage
            loss = -torch.min(surr1, surr2) + F.smooth_l1_loss(self.v(s), td_target.detach())
            self.optimizer.zero_grad()
            loss.mean().backward()
            self.optimizer.step()

## 3. Train

Data is collected for `T_horizon` steps, then trained on `K_epoch` times. **1,200 episodes**
is enough to watch PPO climb quickly (a few minutes on CPU).

In [ ]:
EPISODES = 1200
env = gym.make('CartPole-v1')
model = PPO()
returns = []; score = 0.0
for n_epi in range(EPISODES):
    s, _ = env.reset()
    done = False; ep_ret = 0.0
    while not done:
        for t in range(T_horizon):
            prob = model.pi(torch.from_numpy(s).float())
            a = Categorical(prob).sample().item()
            s2, r, term, trunc, _ = env.step(a)
            done = term or trunc
            model.put_data((s, a, r/100.0, s2, prob[a].item(), done))
            s = s2; ep_ret += r
            if done: break
        model.train_net()
    returns.append(ep_ret); score += ep_ret
    if n_epi % 20 == 0 and n_epi != 0:
        print(f'# episode {n_epi:5d}   avg score (last 20): {score/20:6.1f}'); score = 0.0
env.close()

## 4. The learning curve

Expect the **fastest early climb** of the three policy-gradient methods. On CartPole PPO also
wobbles — the task is too easy to show its stability advantage, which appears on hard,
high-dimensional problems (robotics, Atari, RLHF).

In [ ]:
import numpy as np
r = np.array(returns); w = 50
ma = np.convolve(r, np.ones(w)/w, mode='valid')
plt.figure(figsize=(8,4.5))
plt.plot(r, color='#bfdbfe', lw=0.8, label='episode return')
plt.plot(np.arange(w-1, len(r)), ma, color='#2563eb', lw=2.2, label=f'{w}-ep moving avg')
plt.axhline(500, color='#111827', ls='--', lw=1.2)
plt.xlabel('episode'); plt.ylabel('return'); plt.title('PPO on CartPole-v1')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Where to go next

- **Tune the clip.** Try `eps_clip` of 0.1 vs 0.2, or `K_epoch` of 3 vs 10 — more reuse is more
  sample-efficient but riskier.
- **Continuous actions.** The next topic swaps the softmax policy for a Gaussian so PPO can
  control continuous action spaces — where its stability really matters.